In [6]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Find the project root that contains the src folder
current = Path.cwd().resolve()

for p in [current] + list(current.parents):
    if (p / "src").exists():
        repo_root = p
        break
else:
    raise FileNotFoundError("Could not find a folder containing 'src'.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Current working directory:", current)
print("Added repo root:", repo_root)
print("src exists:", (repo_root / "src").exists())

from pathlib import Path
import numpy as np
import pandas as pd

import src.utils.pdata_io as pdio
from src.proc.extract_epoch_windows import load_epoch_windows

data_root, pdata_root, cc_data = pdio.load_project_context()

windows_df = load_epoch_windows(
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

valid_windows = windows_df[windows_df["valid_window"]].copy()

print("All windows:", windows_df.shape)
print("Valid windows:", valid_windows.shape)

valid_windows.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Current working directory: /home/nmldata2/ccaw/Python/notebooks
Added repo root: /home/nmldata2/ccaw/Python
src exists: True
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
[LOADED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s
All windows: (103980, 29)
Valid windows: (83023, 29)


,phase,epoch_name,n_windows
0,air_training,air_off_mid_post_1s,3379
1,air_training,air_off_mid_pre_1s,3379
2,air_training,air_off_post_1s,2838
3,air_training,air_off_pre_1s,2454
4,air_training,air_on_mid_post_1s,1762
5,air_training,air_on_mid_pre_1s,1134
6,air_training,air_on_post_1s,2465
7,air_training,air_on_pre_1s,3225
8,air_training,pseudo_tone_off_post_1s,1150
9,air_training,pseudo_tone_off_pre_1s,1762


In [5]:
valid_windows

,animal,date,phase,event_number,anchor_name,anchor_type,anchor_time_s,window_position,window_s,epoch_name,...,session_duration_s,session_time_bin,parent_event_valid,valid_window_basic,valid_window,overlap_flag,overlap_s,overlap_with,anchor_warning,invalid_reason
12,NML_04,2026_01_12,air_training,1,pseudo_tone_on,pseudo,16.956400,post,1.0,pseudo_tone_on_post_1s,...,1308.9,early,True,True,True,False,0.0,,,
13,NML_04,2026_01_12,air_training,1,pseudo_tone_on,pseudo,16.956400,pre,1.0,pseudo_tone_on_pre_1s,...,1308.9,early,True,True,True,False,0.0,,,
14,NML_04,2026_01_12,air_training,1,air_on,main,19.956400,post,1.0,air_on_post_1s,...,1308.9,early,True,True,True,False,0.0,,,
15,NML_04,2026_01_12,air_training,1,air_on,main,19.956400,pre,1.0,air_on_pre_1s,...,1308.9,early,True,True,True,False,0.0,,,
20,NML_04,2026_01_12,air_training,1,air_off,main,23.979000,post,1.0,air_off_post_1s,...,1308.9,early,True,True,True,False,0.0,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103963,NML_08,2026_03_21,tone_air_training,47,air_on_mid,middle,1194.077209,pre,1.0,air_on_mid_pre_1s,...,1232.7,late,True,True,True,False,0.0,,,
103964,NML_08,2026_03_21,tone_air_training,47,air_off,main,1198.325439,post,1.0,air_off_post_1s,...,1232.7,late,True,True,True,False,0.0,,,
103965,NML_08,2026_03_21,tone_air_training,47,air_off,main,1198.325439,pre,1.0,air_off_pre_1s,...,1232.7,late,True,True,True,False,0.0,,,
103966,NML_08,2026_03_21,tone_air_training,47,air_off_mid,middle,1207.316895,post,1.0,air_off_mid_post_1s,...,1232.7,late,True,True,True,False,0.0,,,


In [4]:

from src.proc.behavior_metrics import compute_encoder_metrics_for_windows


encoder_epoch_df = compute_encoder_metrics_for_windows(
    valid_windows,
    speed_thresh=0.5
)

encoder_epoch_df.head()

,animal,date,phase,event_number,anchor_name,anchor_type,anchor_time_s,window_position,window_s,epoch_name,...,max_speed_net_cms,distance_path_cm,distance_net_cm,net_direction_bias,frac_stationary,frac_moving,frac_forward,frac_backward,frac_low_net_movement,dominant_locomotor_state
0,NML_04,2026_01_12,air_training,1,air_off_mid,middle,31.5222,post,1.0,air_off_mid_post_1s,...,5.780530,2.111150,1.407434,0.680638,0.0808,0.9192,0.6118,0.3074,0.0,forward
1,NML_04,2026_01_12,air_training,1,air_off_mid,middle,31.5222,pre,1.0,air_off_mid_pre_1s,...,5.277876,1.482832,1.130973,0.755565,0.3230,0.6770,0.4804,0.1966,0.0,forward
2,NML_04,2026_01_12,air_training,1,air_off,main,23.9790,post,1.0,air_off_post_1s,...,14.074335,8.721061,8.721061,1.000000,0.0000,1.0000,1.0000,0.0000,0.0,forward
3,NML_04,2026_01_12,air_training,1,air_off,main,23.9790,pre,1.0,air_off_pre_1s,...,13.571680,5.101946,4.850619,0.951624,0.0340,0.9660,0.8530,0.1130,0.0,forward
4,NML_04,2026_01_12,air_training,1,air_on,main,19.9564,post,1.0,air_on_post_1s,...,0.753982,0.402124,-0.201062,-0.487133,0.5560,0.4440,0.0924,0.3516,0.0,stationary


In [ ]:
# encoder_epoch_df = pd.read_hdf(
#     Path(pdata_root) / "_cache" / "behavior_epoch_metrics.h5",
#     key="encoder/prepost_1s_speedThresh_1cms"
# )

from src.proc.behavior_metrics import add_normalized_phase_day

encoder_epoch_df = add_normalized_phase_day(encoder_epoch_df)
encoder_epoch_df.head()

,animal,date,phase,event_number,anchor_name,anchor_type,anchor_time_s,window_position,window_s,epoch_name,...,net_direction_bias,frac_stationary,frac_moving,frac_forward,frac_backward,frac_low_net_movement,dominant_locomotor_state,phase_session_number,n_sessions_in_phase,normalized_phase_day
0,NML_04,2026_01_12,air_training,1,air_off_mid,middle,31.5222,post,1.0,air_off_mid_post_1s,...,0.680638,0.0808,0.9192,0.6118,0.3074,0.0,forward,1.0,15.0,0.0
1,NML_04,2026_01_12,air_training,1,air_off_mid,middle,31.5222,pre,1.0,air_off_mid_pre_1s,...,0.755565,0.3230,0.6770,0.4804,0.1966,0.0,forward,1.0,15.0,0.0
2,NML_04,2026_01_12,air_training,1,air_off,main,23.9790,post,1.0,air_off_post_1s,...,1.000000,0.0000,1.0000,1.0000,0.0000,0.0,forward,1.0,15.0,0.0
3,NML_04,2026_01_12,air_training,1,air_off,main,23.9790,pre,1.0,air_off_pre_1s,...,0.951624,0.0340,0.9660,0.8530,0.1130,0.0,forward,1.0,15.0,0.0
4,NML_04,2026_01_12,air_training,1,air_on,main,19.9564,post,1.0,air_on_post_1s,...,-0.487133,0.5560,0.4440,0.0924,0.3516,0.0,stationary,1.0,15.0,0.0


In [8]:
cache_dir = Path(pdata_root) / "_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

encoder_metrics_file = cache_dir / "behavior_epoch_metrics.h5"

encoder_epoch_df.to_hdf(
    encoder_metrics_file,
    key="encoder/prepost_1s_speedThresh_1cms",
    mode="w",
    format="table"
)

print("Saved:", encoder_metrics_file)

Saved: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_metrics.h5


In [6]:
encoder_epoch_df.groupby(
    ["phase", "epoch_name"]
).size().reset_index(name="n_windows")

,phase,epoch_name,n_windows
0,air_training,air_off_mid_post_1s,3379
1,air_training,air_off_mid_pre_1s,3379
2,air_training,air_off_post_1s,2838
3,air_training,air_off_pre_1s,2454
4,air_training,air_on_mid_post_1s,1762
5,air_training,air_on_mid_pre_1s,1134
6,air_training,air_on_post_1s,2465
7,air_training,air_on_pre_1s,3225
8,air_training,pseudo_tone_off_post_1s,1150
9,air_training,pseudo_tone_off_pre_1s,1762


In [7]:
speed_summary = (
    encoder_epoch_df
    .groupby(["phase", "epoch_name"])
    .agg(
        mean_path_speed=("mean_speed_path_cms", "mean"),
        sem_path_speed=("mean_speed_path_cms", lambda x: x.std() / np.sqrt(len(x))),
        mean_net_speed=("mean_speed_net_cms", "mean"),
        n=("mean_speed_path_cms", "count"),
    )
    .reset_index()
)

speed_summary

,phase,epoch_name,mean_path_speed,sem_path_speed,mean_net_speed,n
0,air_training,air_off_mid_post_1s,2.021128,0.053808,1.963205,3379
1,air_training,air_off_mid_pre_1s,2.158887,0.054162,2.095409,3379
2,air_training,air_off_post_1s,4.334152,0.212683,3.580176,2838
3,air_training,air_off_pre_1s,6.035266,0.048383,5.964671,2454
4,air_training,air_on_mid_post_1s,4.521962,0.068697,4.401002,1762
5,air_training,air_on_mid_pre_1s,3.528038,0.082097,3.401191,1134
6,air_training,air_on_post_1s,2.632072,0.038650,2.326716,2465
7,air_training,air_on_pre_1s,1.428689,0.050584,1.370447,3225
8,air_training,pseudo_tone_off_post_1s,3.810288,0.086504,3.709521,1150
9,air_training,pseudo_tone_off_pre_1s,4.001601,0.071583,3.884586,1762


In [8]:
state_summary = (
    encoder_epoch_df
    .groupby(["phase", "epoch_name"])
    .agg(
        frac_stationary=("frac_stationary", "mean"),
        frac_moving=("frac_moving", "mean"),
        frac_forward=("frac_forward", "mean"),
        frac_backward=("frac_backward", "mean"),
        frac_low_net=("frac_low_net_movement", "mean"),
        n=("frac_stationary", "count"),
    )
    .reset_index()
)

state_summary

,phase,epoch_name,frac_stationary,frac_moving,frac_forward,frac_backward,frac_low_net,n
0,air_training,air_off_mid_post_1s,0.607812,0.392188,0.371419,0.020769,0.0,3379
1,air_training,air_off_mid_pre_1s,0.578787,0.421213,0.399322,0.021891,0.0,3379
2,air_training,air_off_post_1s,0.072125,0.927875,0.830184,0.097692,0.0,2838
3,air_training,air_off_pre_1s,0.018368,0.981632,0.963370,0.018262,0.0,2454
4,air_training,air_on_mid_post_1s,0.184558,0.815442,0.782435,0.033007,0.0,1762
5,air_training,air_on_mid_pre_1s,0.286352,0.713648,0.680032,0.033616,0.0,1134
6,air_training,air_on_post_1s,0.231413,0.768587,0.657500,0.111088,0.0,2465
7,air_training,air_on_pre_1s,0.717806,0.282194,0.263233,0.018961,0.0,3225
8,air_training,pseudo_tone_off_post_1s,0.256447,0.743553,0.710656,0.032897,0.0,1150
9,air_training,pseudo_tone_off_pre_1s,0.189851,0.810149,0.767818,0.042330,0.0,1762
